# CUDA Transformer Inference Benchmark — run everything on Colab

1. `Runtime → Change runtime type → GPU` (T4 is fine), then run the cells in order.
2. The run builds the project, runs **all correctness tests first** (it stops if any fails), then every benchmark and the PyTorch baseline.
3. Results land in `benchmarks/<GPU>/` and `summary.md` is shown at the end. Download the zip and copy the numbers into `docs/12_results.md`.

In [ ]:
!nvidia-smi
!nvcc --version

## Get the code — use ONE of the two cells below

**Option A (git):** put your repository URL in the first cell.

**Option B (zip):** on your computer, zip the `cuda-transformer-performance` folder itself (so the zip contains that folder at its top level) and upload it with the second cell.

In [ ]:
# Option A: clone from git (replace the URL)
REPO_URL = "https://github.com/<your-user>/cuda-transformer-performance.git"
!git clone {REPO_URL} /content/cuda-transformer-performance

In [ ]:
# Option B: upload cuda-transformer-performance.zip
from google.colab import files
uploaded = files.upload()
zip_name = next(iter(uploaded))
!unzip -q -o "{zip_name}" -d /content/

In [ ]:
# Find the project wherever it was cloned/unzipped (it may be nested, e.g. /content/<folder>/cuda-transformer-performance),
# using the most recently extracted copy.
import glob, os
roots = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/**/scripts/run_all.sh', recursive=True)]
if not roots:
    raise SystemExit('No project with scripts/run_all.sh under /content: upload (or clone) the current project first.')
root = max(roots, key=lambda r: os.path.getmtime(os.path.join(r, 'scripts', 'run_all.sh')))
os.chdir(root)
print('Project:', root)
# Files edited on Windows may have CRLF line endings, which break bash scripts on Linux.
!sed -i 's/\r$//' scripts/*.sh
!bash scripts/run_all.sh

In [ ]:
# Show the generated summary
import glob
from IPython.display import Markdown, display
summary = sorted(glob.glob("benchmarks/*/summary.md"))[-1]
print(summary)
display(Markdown(open(summary).read()))

In [ ]:
# Download all results (raw text, CSV, summary)
!zip -q -r results.zip benchmarks
from google.colab import files
files.download("results.zip")